# 03 — Political Layer Analysis

FEC data explorer and committee accountability.

In [ ]:
import json, pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "political"

fec_path = DATA / "fec_funding_profiles.json"
if not fec_path.exists():
    print("FEC data not found. Run: python fec_integration_v251d.py")
else:
    with open(fec_path) as f:
        raw = json.load(f)
    profiles = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    fec = pd.DataFrame(profiles)
    print(f"{len(fec)} member profiles loaded")
    print(fec[["name","state","total_receipts","business_total","labor_total"]].to_string())


## Business vs Labor Contributions

In [ ]:
LAYOUT = dict(paper_bgcolor="#121212", plot_bgcolor="#1e1e1e",
              font=dict(color="#e8e8e8", family="monospace"))
if "fec" in dir():
    fig = go.Figure()
    s = fec.sort_values("business_total", ascending=False)
    fig.add_trace(go.Bar(name="Business", x=s["name"], y=s.get("business_total", []),
                          marker_color="#F39C12"))
    fig.add_trace(go.Bar(name="Labor", x=s["name"], y=s.get("labor_total", []),
                          marker_color="#4aa8d8"))
    fig.update_layout(**LAYOUT, barmode="group",
                      title="Business vs Labor Contributions (2024 Cycle)")
    fig.show()


## UI Committee Members

In [ ]:
report_path = DATA / "political_layer_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    ui_members = pd.DataFrame(report.get("ui_members_detail", []))
    if not ui_members.empty:
        print(f"UI-relevant committee members: {len(ui_members)}")
        for _, m in ui_members.iterrows():
            income = m.get("constituent_median_income")
            print(f"  {m['name']} ({m['state']}, {m['chamber']}) income: ${income:,}" if income else f"  {m['name']} ({m['state']})")


## Cross-tab: Committee × Employer Contributions

In [ ]:
if "fec" in dir() and "ui_members" in dir() and not ui_members.empty:
    ui_names = set(ui_members["name"].str.lower())
    fec["on_ui_committee"] = fec["name"].str.lower().apply(lambda n: any(u in n for u in ui_names))
    print("\nMean employer contributions:")
    print(fec.groupby("on_ui_committee")[["business_total","total_receipts"]].mean().to_string())
